# Tuần 2 — PhoBERT Multi-task Training
**ABSA VLSP 2018 Hotel | NLP Course — HUST**

Chạy trên **Google Colab (T4 GPU)** hoặc **Kaggle (T4 x2)**.

| Bước | Mô tả |
|------|-------|
| Cell 1 | Check GPU + Install dependencies |
| Cell 2 | Mount Drive / Set working dir |
| Cell 3 | Download data nếu chưa có |
| Cell 4 | Preprocessing (dùng cache nếu có) |
| Cell 5 | Kiểm tra EDA config |
| Cell 6 | **TRAIN** (main cell) |
| Cell 7 | Ablation: cls_only |
| Cell 8 | Learning curve |
| Cell 9 | Đọc summary report |

In [ ]:
# Cell 1 — Check GPU & Install
import torch, os

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
else:
    print("⚠️  NONE — check runtime! Runtime → Change runtime type → T4 GPU")

!pip install -q transformers==4.38.0 underthesea tabulate tqdm scikit-learn

In [ ]:
# Cell 2 — Mount & Set working dir
import os, sys

# ── COLAB ──────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = "/content/drive/MyDrive/absa-vlsp2018-hotel"
os.chdir(PROJECT_DIR)

# ── KAGGLE (uncomment nếu dùng Kaggle) ────────────────────────────────────────
# !git clone https://github.com/YOUR_REPO/absa-vlsp2018-hotel /kaggle/working/project --depth=1
# os.chdir("/kaggle/working/project")

sys.path.insert(0, "code/week1")
sys.path.insert(0, "code/week2")
print(f"Working dir: {os.getcwd()}")
!ls

In [ ]:
# Cell 3 — Download data nếu chưa có
import pandas as pd

if not os.path.exists("data/train.csv"):
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/
    print("Data downloaded ✓")
else:
    print("Data exists ✓")

df = pd.read_csv("data/train.csv")
print(f"Train: {len(df)} rows × {df.shape[1]} cols")

In [ ]:
# Cell 4 — Preprocessing (dùng cache nếu có)
if not os.path.exists("data/train_preprocessed.csv"):
    print("Chưa có cache — chạy preprocessing...")
    from step3_preprocessing import preprocess_dataframe, VnCoreNLPSegmenter
    segmenter = VnCoreNLPSegmenter(use_fallback=True)  # underthesea fallback
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(f"data/{split}.csv")
        preprocess_dataframe(
            df, segmenter=segmenter,
            cache_path=f"data/{split}_preprocessed.csv"
        )
        print(f"  {split}: done")
    segmenter.close()
else:
    print("Cache exists ✓ (data/*_preprocessed.csv)")

# Kiểm tra nhanh
sample = pd.read_csv("data/train_preprocessed.csv").iloc[0]
print(f"\nSample original : {sample['Review'][:80]}")
print(f"Sample processed: {sample['processed_review'][:80]}")

In [ ]:
# Cell 5 — Kiểm tra EDA config
import json

enc_cfg = json.load(open("outputs/eda/encoder_config.json"))
cw      = json.load(open("outputs/eda/class_weights.json"))

print("Encoder config:")
print(json.dumps(enc_cfg, indent=2))
print(f"\nGlobal class weights:")
for cls, w in cw['global_weights'].items():
    label = {"0": "absent", "1": "positive", "2": "negative", "3": "neutral"}.get(cls, cls)
    print(f"  {label}: {w:.2f}x {'← sẽ clip tại 10.0' if w > 10 else ''}")

In [ ]:
# Cell 6 — TRAIN (Main cell)
# concat_4_layers: SOTA architecture (3072 dim, ~1-2% F1 tốt hơn cls_only)
from run_experiment import main

test_metrics = main(encoder_option="concat_4_layers")

In [ ]:
# Cell 7 — Ablation: cls_only (cho báo cáo)
# Chạy sau khi Cell 6 hoàn tất để so sánh
test_metrics_cls = main(encoder_option="cls_only")

print("\n=== ABLATION SUMMARY ===")
print(f"concat_4_layers: {test_metrics['macro_combined_f1']:.4f}")
print(f"cls_only:        {test_metrics_cls['macro_combined_f1']:.4f}")
gain = test_metrics['macro_combined_f1'] - test_metrics_cls['macro_combined_f1']
print(f"Gain:            {gain * 100:+.2f}%")

In [ ]:
# Cell 8 — Learning Curve (BẮT BUỘC cho báo cáo)
import json
import matplotlib.pyplot as plt

history  = json.load(open("outputs/results/training_history.json"))
best_ep  = history["best_epoch"]
epochs   = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs, history["train_loss"], "o-", color="crimson",   label="Train Loss")
ax1.plot(epochs, history["dev_loss"],   "o-", color="steelblue", label="Dev Loss")
ax1.axvline(best_ep, color="green", ls="--", alpha=0.7, label=f"Best epoch ({best_ep})")
ax1.set(title="Loss per Epoch", xlabel="Epoch", ylabel="Loss")
ax1.legend(); ax1.grid(alpha=0.3)

# F1
ax2.plot(epochs, history["dev_acd_f1"],      "o-", color="darkorange", label="Dev ACD F1")
ax2.plot(epochs, history["dev_spc_f1"],      "o-", color="purple",     label="Dev SPC F1")
ax2.plot(epochs, history["dev_combined_f1"], "o-", color="green",      label="Dev Combined F1", lw=2)
ax2.axvline(best_ep, color="green",  ls="--", alpha=0.7, label=f"Best ({best_ep})")
ax2.axhline(0.7732,  color="red",    ls=":",  alpha=0.6, label="SOTA 0.7732")
ax2.set(title="F1 per Epoch", xlabel="Epoch", ylabel="F1")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/eda/learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n📊 Phân tích learning curve:")
print(f"  Best epoch: {best_ep}")
print(f"  Best Combined F1: {history['best_combined_f1']:.4f}")
if len(history["dev_loss"]) > best_ep:
    print(f"  Overfitting bắt đầu từ epoch {best_ep + 1}:")
    print(f"    Dev loss: {history['dev_loss'][best_ep-1]:.4f} → {history['dev_loss'][best_ep]:.4f}")
    print(f"    Dev F1 không tăng thêm dù train loss tiếp tục giảm")

In [ ]:
# Cell 9 — Đọc summary report
with open("outputs/results/week2_summary.md", encoding="utf-8") as f:
    print(f.read())